In [1]:
import pandas as pd
from pathlib import Path

RAW = Path("")
OUT = Path("")
OUT.mkdir(parents=True, exist_ok=True)

# ── Constants ─────────────────────────────────────────────────────────────────
TIER1 = {
    "BLZ","CRI","CUB","DOM","SLV","GTM","GUY",
    "HTI","HND","JAM","NIC","PAN","SUR","TTO","BHS","BRB"
    # PRI excluded — not in INFORM; handled via FEMA data
}
PRESSURE = {"COL", "VEN"}
ALL_COUNTRIES = TIER1 | PRESSURE

YEARS = list(range(2019, 2025))  # 2019–2024 inclusive

# INFORM indicator IDs → our column names
INDICATOR_MAP = {
    "INFORM": "inform_risk_score",
    "HA":     "hazard_exposure",
    "VU":     "vulnerability",
    "CC":     "lack_coping_capacity",
    "HA.NAT": "natural_hazard",
    "VU.SEV": "socioeconomic_vulnerability",
    "CC.INF": "infrastructure_coping",
}


def main():
    print("Loading INFORM trend file...")
    df = pd.read_excel(
        RAW / "INFORM2024_TREND_2015_2024_v70_ALL.xlsx",
        sheet_name="INFORM2025_2nd_edition_Trend"
    )

    # ── Filter to our countries, indicators, and years ────────────────────────
    df = df[
        df["Iso3"].isin(ALL_COUNTRIES) &
        df["IndicatorId"].isin(INDICATOR_MAP.keys()) &
        df["INFORMYear"].isin(YEARS)
    ].copy()

    df = df.rename(columns={"Iso3": "iso3", "INFORMYear": "year"})
    df["col_name"] = df["IndicatorId"].map(INDICATOR_MAP)
    df["IndicatorScore"] = pd.to_numeric(df["IndicatorScore"], errors="coerce")

    # ── Pivot to wide format: one row per country × year ─────────────────────
    wide = df.pivot_table(
        index=["iso3", "year"],
        columns="col_name",
        values="IndicatorScore"
    ).reset_index()
    wide.columns.name = None

    # Ensure column order
    score_cols = list(INDICATOR_MAP.values())
    wide = wide[["iso3", "year"] + score_cols]
    wide["inform_data_flag"] = "inform_annual"

    # ── QA: coverage check ────────────────────────────────────────────────────
    print("\n=== QA SUMMARY ===")
    print(f"Rows: {len(wide)} (expected {len(ALL_COUNTRIES) * len(YEARS)} = {len(ALL_COUNTRIES)}×{len(YEARS)})")
    print(f"Nulls per column:\n{wide[score_cols].isna().sum().to_string()}")
    print(f"\nPRI absent (expected): {'PRI' not in wide['iso3'].unique()}")
    print(f"VEN present: {'VEN' in wide['iso3'].unique()}")

    print("\nINFORM Risk scores 2024 (sorted by risk):")
    y24 = wide[wide["year"]==2024][["iso3","inform_risk_score"]].sort_values(
        "inform_risk_score", ascending=False
    )
    print(y24.to_string(index=False))

    # ── Write country-year QA file ────────────────────────────────────────────
    wide.to_csv(OUT / "inform_country_year.csv", index=False)
    print(f"\n  Wrote inform_country_year.csv  ({len(wide)} rows)")

    # ── Expand to country-month ───────────────────────────────────────────────
    # Each annual score is repeated across all 12 months of that year.
    # Monthly variation is zero by construction — see methodology note above.
    rows = []
    for _, r in wide.iterrows():
        for month in pd.date_range(f"{int(r.year)}-01-01", f"{int(r.year)}-12-01", freq="MS"):
            row = {"iso3": r.iso3, "year": int(r.year), "month": month}
            for col in score_cols:
                row[col] = r[col]
            row["inform_data_flag"] = "inform_annual"
            rows.append(row)

    monthly = pd.DataFrame(rows)
    monthly["month"] = pd.to_datetime(monthly["month"])
    monthly = monthly.sort_values(["iso3", "month"]).reset_index(drop=True)

    print(f"  Rows in monthly file: {len(monthly)} (expected {len(ALL_COUNTRIES) * len(YEARS) * 12} = {len(ALL_COUNTRIES)}×{len(YEARS)}×12)")

    out_cols = ["iso3", "year", "month"] + score_cols + ["inform_data_flag"]
    monthly[out_cols].to_csv(OUT / "inform_country_month.csv", index=False)
    print(f"  Wrote inform_country_month.csv ({len(monthly)} rows)")

    # ── Coverage reminder ─────────────────────────────────────────────────────
    print("\n=== TIER 1 COVERAGE ===")
    covered = sorted(set(monthly["iso3"].unique()) & TIER1)
    not_covered = sorted(TIER1 - set(monthly["iso3"].unique()))
    print(f"  WITH INFORM data    : {covered}")
    print(f"  WITHOUT INFORM data : {not_covered}")
    print(f"  → PRI uses FEMA-derived vulnerability proxy in master grid")


if __name__ == "__main__":
    main()

Loading INFORM trend file...

=== QA SUMMARY ===
Rows: 108 (expected 108 = 18×6)
Nulls per column:
inform_risk_score              0
hazard_exposure                0
vulnerability                  0
lack_coping_capacity           0
natural_hazard                 0
socioeconomic_vulnerability    0
infrastructure_coping          0

PRI absent (expected): True
VEN present: True

INFORM Risk scores 2024 (sorted by risk):
iso3  inform_risk_score
 HTI                7.0
 VEN                6.4
 HND                5.5
 COL                5.2
 GTM                5.1
 NIC                4.9
 DOM                4.5
 PAN                4.1
 SLV                4.1
 GUY                3.9
 BLZ                3.7
 CUB                3.3
 CRI                3.2
 SUR                3.1
 TTO                3.0
 JAM                2.9
 BHS                2.3
 BRB                2.1

  Wrote inform_country_year.csv  (108 rows)
  Rows in monthly file: 1296 (expected 1296 = 18×6×12)
  Wrote inform_country_m